In [1]:
import pandas as pd
import re
import ast
from pathlib import Path

In [2]:
BASE_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline")

CV_INPUT_PATH = BASE_DIR / "data_outputs" / "step02_sections" / "02_cv_sectioned.xlsx"
OUTPUT_DIR = BASE_DIR / "data_outputs" / "step03_skill_extract"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "03_cv_skills_extracted.xlsx"

# Sửa 2 path này đúng chỗ file của bạn
SKILL_MAPPING_PATH = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/ESCO_taxonomy/notebook_clean/14_skill_mapping_clean.xlsx")
DJINNI_PATH = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/Djinni/notebook_clean/08_djinni_step4_final.xlsx")

print("CV input:", CV_INPUT_PATH, CV_INPUT_PATH.exists())
print("Skill mapping:", SKILL_MAPPING_PATH, SKILL_MAPPING_PATH.exists())
print("Djinni:", DJINNI_PATH, DJINNI_PATH.exists())
print("Output:", OUTPUT_PATH)

CV input: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step02_sections/02_cv_sectioned.xlsx True
Skill mapping: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/ESCO_taxonomy/notebook_clean/14_skill_mapping_clean.xlsx True
Djinni: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/Djinni/notebook_clean/08_djinni_step4_final.xlsx True
Output: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step03_skill_extract/03_cv_skills_extracted.xlsx


In [3]:
cv_df = pd.read_excel(CV_INPUT_PATH, engine="openpyxl")
skill_df = pd.read_excel(SKILL_MAPPING_PATH, engine="openpyxl")
djinni_df = pd.read_excel(DJINNI_PATH, engine="openpyxl")

print("cv_df:", cv_df.shape)
print("skill_df:", skill_df.shape)
print("djinni_df:", djinni_df.shape)

display(cv_df.head(2))
display(skill_df.head(2))
display(djinni_df.head(2))

cv_df: (20, 20)
skill_df: (1171, 10)
djinni_df: (1171, 28)


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean,section_summary,section_skills,section_experience,section_education,section_projects,section_certifications,section_other,section_summary_len,section_skills_len,section_experience_len,section_education_len,section_projects_len,section_certifications_len,section_other_len
0,C001,Backend Developer with 2 years of experience b...,backend developer with 2 years of experience b...,279,270,False,NaN,NaN,building restful apis using java spring boot h...,NaN,NaN,NaN,backend developer with 2 years of,0,0,225,0,0,0,33
1,C002,Frontend Developer with strong experience in R...,frontend developer with strong experience in r...,258,249,False,NaN,NaN,in reactjs typescript javascript html5 css3 re...,NaN,NaN,NaN,frontend developer with strong,0,0,207,0,0,0,30


,skill_id,skill_name,skill_type,group,relation_count,skill_group,skill_subgroup,mapped_taxonomy_group,mapped_taxonomy_subgroup,notes
0,NaN,Python (computer programming),optional,extended,20,AI / Data Tool,Machine Learning / AI,Data & AI,AI / Machine Learning,NaN
1,NaN,computer vision,optional,core,1,AI / Data Tool,Machine Learning / AI,Data & AI,AI / Machine Learning,NaN


,row_id,nhom_lon,nhom_nho,cum_ky_nang,skill_subgroup,ten_goc,ten_sach,ten_ngoai_thi_truong,cac_ten_gan_giong,huong_xu_ly,...,buoc,ten_thi_truong,ten_gan_giong,flag_missing_market_name,flag_missing_alias,market_name_norm,flag_duplicate_market_name,review_priority,review_issue,can_xem_thu_cong
0,2,Data & AI,AI / Machine Learning,AI / Data Tool,Machine Learning / AI,Python (computer programming),python (computer programming),python (computer programming),NaN,giu_nguyen,...,4,python (computer programming),NaN,0,0,python (computer programming),0,NaN,NaN,0
1,3,Data & AI,AI / Machine Learning,AI / Data Tool,Machine Learning / AI,computer vision,computer vision,computer vision,NaN,giu_nguyen,...,4,computer vision,NaN,0,0,computer vision,0,NaN,NaN,0


In [4]:
required_cv_cols = [
    "candidate_id",
    "section_skills",
    "section_experience",
    "section_projects",
    "section_other"
]

missing_cv_cols = [col for col in required_cv_cols if col not in cv_df.columns]
if missing_cv_cols:
    raise ValueError(f"Thiếu cột trong CV sectioned file: {missing_cv_cols}")

print("CV input đủ cột để extract skill.")

CV input đủ cột để extract skill.


In [5]:
print("skill_df columns:")
print(skill_df.columns.tolist())

print("\ndjinni_df columns:")
print(djinni_df.columns.tolist())

skill_df columns:
['skill_id', 'skill_name', 'skill_type', 'group', 'relation_count', 'skill_group', 'skill_subgroup', 'mapped_taxonomy_group', 'mapped_taxonomy_subgroup', 'notes']

djinni_df columns:
['row_id', 'nhom_lon', 'nhom_nho', 'cum_ky_nang', 'skill_subgroup', 'ten_goc', 'ten_sach', 'ten_ngoai_thi_truong', 'cac_ten_gan_giong', 'huong_xu_ly', 'ghi_chu_djinni', 'skill_id', 'skill_type', 'group', 'relation_count', 'notes', 'da_kiem', 'vong', 'buoc', 'ten_thi_truong', 'ten_gan_giong', 'flag_missing_market_name', 'flag_missing_alias', 'market_name_norm', 'flag_duplicate_market_name', 'review_priority', 'review_issue', 'can_xem_thu_cong']


In [6]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def normalize_skill_phrase(text):
    text = normalize_text(text)
    text = re.sub(r"[^\w\s\+\#\.\-/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [7]:
possible_skill_name_cols = [
    "ten_sach",
    "skill_clean",
    "preferred_label",
    "skill_name",
    "skill",
    "ten_goc"
]

skill_name_col = None
for col in possible_skill_name_cols:
    if col in skill_df.columns:
        skill_name_col = col
        break

print("skill_name_col =", skill_name_col)

if skill_name_col is None:
    raise ValueError("Không tìm thấy cột tên skill chuẩn trong file 14_skill_mapping_clean.xlsx")

skill_name_col = skill_name


In [8]:
canonical_skills = (
    skill_df[skill_name_col]
    .dropna()
    .astype(str)
    .map(normalize_skill_phrase)
)

canonical_skills = sorted(set([s for s in canonical_skills if s]))
print("Số canonical skills:", len(canonical_skills))
print(canonical_skills[:30])

Số canonical skills: 1171
['3d lighting', '3d modelling', '3d printing process', '3d texturing', 'abap', 'absorb learning management systems', 'accounting techniques', 'acquire system component', 'adapt developed game to the market', 'adapt to changes in technological development plans', 'adapt to changing situations', 'address problems critically', 'adjust ict system capacity', 'administer ict system', 'adobe photoshop', 'advertising techniques', 'advice on security risk management', 'advise client on technical possibilities', 'advise on communication strategies', 'advise on efficiency improvements', 'advise on financial matters', 'advise on organisational culture', 'advise on personnel management', 'advise on risk management', 'advise on safety improvements', 'advise on tax policy', 'agile development', 'agile project management', 'ajax', 'ajax framework']


In [9]:
possible_djinni_canonical_cols = ["ten_sach", "skill_clean", "skill_name", "ten_goc"]
djinni_canonical_col = None
for col in possible_djinni_canonical_cols:
    if col in djinni_df.columns:
        djinni_canonical_col = col
        break

print("djinni_canonical_col =", djinni_canonical_col)

possible_alias_cols = ["ten_ngoai_thi_truong", "cac_ten_gan_giong", "ten_goc"]
djinni_alias_cols = [col for col in possible_alias_cols if col in djinni_df.columns]

print("djinni_alias_cols =", djinni_alias_cols)

djinni_canonical_col = ten_sach
djinni_alias_cols = ['ten_ngoai_thi_truong', 'cac_ten_gan_giong', 'ten_goc']


In [10]:
def parse_alias_value(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    if not value:
        return []

    # thử parse dạng list string
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    # split theo ; hoặc ,
    parts = re.split(r"[;,|]", value)
    return [p.strip() for p in parts if p.strip()]

In [11]:
alias_to_skill = {}

if djinni_canonical_col is not None:
    for _, row in djinni_df.iterrows():
        canonical = normalize_skill_phrase(row.get(djinni_canonical_col, ""))
        if not canonical:
            continue

        # tự map chính canonical luôn
        alias_to_skill[canonical] = canonical

        for alias_col in djinni_alias_cols:
            raw_value = row.get(alias_col, "")
            alias_list = parse_alias_value(raw_value)

            # với cột đơn trị như ten_ngoai_thi_truong / ten_goc
            if not alias_list and pd.notna(raw_value):
                alias_list = [str(raw_value).strip()]

            for alias in alias_list:
                alias_norm = normalize_skill_phrase(alias)
                if alias_norm:
                    alias_to_skill[alias_norm] = canonical

print("Số alias trong Djinni:", len(alias_to_skill))
list(alias_to_skill.items())[:20]

Số alias trong Djinni: 1368


[('python computer programming', 'python computer programming'),
 ('computer vision', 'computer vision'),
 ('deep learning', 'deep learning'),
 ('machine learning', 'machine learning'),
 ('ml', 'machine learning'),
 ('ai', 'machine learning'),
 ('model training', 'machine learning'),
 ('utilise machine learning', 'utilise machine learning'),
 ('frostbite digital game creation systems',
  'frostbite digital game creation systems'),
 ('ict accessibility standards', 'ict accessibility standards'),
 ('advise client on technical possibilities',
  'advise client on technical possibilities'),
 ('analyse big data', 'analyse big data'),
 ('application usability', 'application usability'),
 ('apply statistical analysis techniques',
  'apply statistical analysis techniques'),
 ('assess employees capability levels', 'assess employees capability levels'),
 ('assess financial viability', 'assess financial viability'),
 ('business intelligence', 'business intelligence'),
 ('cloud monitoring and repor

In [12]:
skill_lookup = {}

# canonical từ clean
for skill in canonical_skills:
    skill_lookup[skill] = skill

# alias từ djinni
for alias, canonical in alias_to_skill.items():
    if alias:
        skill_lookup[alias] = canonical

print("Tổng số phrase trong skill lookup:", len(skill_lookup))

Tổng số phrase trong skill lookup: 1368


In [13]:
skill_phrases_sorted = sorted(skill_lookup.keys(), key=lambda x: len(x), reverse=True)

print(skill_phrases_sorted[:50])

['apply research ethics and scientific integrity principles in research activities', 'promote the participation of citizens in scientific and research activities', 'engage local communities in the management of natural protected areas', 'interact professionally in research and professional environments', 'communicate commercial and technical issues in foreign languages', 'draft scientific or academic papers and technical documentation', 'improve customer traveling experiences with augmented reality', 'develop professional network with researchers and scientists', 'manage findable accessible interoperable and reusable data', 'tourist resources of a destination for further development', 'draw sketches to develop textile articles using softwares', 'manage distribution of destination promotional materials', 'assemble health and safety resources in cultural venues', 'integrate marketing strategies with the global strategy', 'organise participation in local or international events', 'collabo

In [14]:
def extract_skills_from_text(text, skill_phrases, lookup_dict):
    text_norm = normalize_skill_phrase(text)
    matched = []

    if not text_norm:
        return []

    for phrase in skill_phrases:
        pattern = r"(?<!\w)" + re.escape(phrase) + r"(?!\w)"
        if re.search(pattern, text_norm):
            matched.append(lookup_dict[phrase])

    # unique nhưng giữ thứ tự
    seen = set()
    result = []
    for skill in matched:
        if skill not in seen:
            seen.add(skill)
            result.append(skill)

    return result

In [15]:
def safe_extract(text):
    return extract_skills_from_text(text, skill_phrases_sorted, skill_lookup)

cv_df["matched_skills_skills_section"] = cv_df["section_skills"].apply(safe_extract)
cv_df["matched_skills_experience_section"] = cv_df["section_experience"].apply(safe_extract)
cv_df["matched_skills_projects_section"] = cv_df["section_projects"].apply(safe_extract)
cv_df["matched_skills_other_section"] = cv_df["section_other"].apply(safe_extract)

In [16]:
def merge_skill_lists(*lists):
    seen = set()
    result = []
    for lst in lists:
        if not isinstance(lst, list):
            continue
        for item in lst:
            if item not in seen:
                seen.add(item)
                result.append(item)
    return result

cv_df["matched_skills_all"] = cv_df.apply(
    lambda row: merge_skill_lists(
        row["matched_skills_skills_section"],
        row["matched_skills_experience_section"],
        row["matched_skills_projects_section"],
        row["matched_skills_other_section"],
    ),
    axis=1
)

cv_df["n_matched_skills"] = cv_df["matched_skills_all"].apply(len)

In [17]:
preview_cols = [
    "candidate_id",
    "section_skills",
    "section_experience",
    "section_projects",
    "section_other",
    "matched_skills_all",
    "n_matched_skills"
]

display(cv_df[preview_cols].head(10))

,candidate_id,section_skills,section_experience,section_projects,section_other,matched_skills_all,n_matched_skills
0,C001,NaN,building restful apis using java spring boot h...,NaN,backend developer with 2 years of,"[perform software unit testing, mysql]",2
1,C002,NaN,in reactjs typescript javascript html5 css3 re...,NaN,frontend developer with strong,"[javascript, typescript, implement frontend we...",3
2,C003,NaN,in python sql excel power bi tableau and panda...,NaN,data analyst with,"[sql, perform data analysis]",2
3,C004,NaN,NaN,NaN,devops engineer familiar with linux docker kub...,[devops],1
4,C005,NaN,in manual testing api testing postman test cas...,NaN,qa engineer with,[],0
5,C006,NaN,NaN,NaN,mobile developer experienced with flutter dart...,[],0
6,C007,NaN,. built text classification pipelines and expe...,NaN,ai engineer with python machine learning sciki...,[machine learning],1
7,C008,NaN,NaN,NaN,cybersecurity analyst with knowledge of networ...,"[ict network security risks, cyber security]",2
8,C009,NaN,in requirement gathering user stories bpmn pro...,NaN,business analyst with,[sql],1
9,C010,NaN,in node.js nestjs expressjs reactjs typescript...,NaN,fullstack developer with,"[postgresql, typescript]",2


In [18]:
cv_df.to_excel(OUTPUT_PATH, index=False)
print("Đã lưu file:", OUTPUT_PATH)

Đã lưu file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step03_skill_extract/03_cv_skills_extracted.xlsx


In [19]:
result = pd.read_excel(OUTPUT_PATH, engine="openpyxl")
print("Output shape:", result.shape)
display(result[["candidate_id", "matched_skills_all", "n_matched_skills"]].head(10))

Output shape: (20, 26)


,candidate_id,matched_skills_all,n_matched_skills
0,C001,"['perform software unit testing', 'mysql']",2
1,C002,"['javascript', 'typescript', 'implement fronte...",3
2,C003,"['sql', 'perform data analysis']",2
3,C004,['devops'],1
4,C005,[],0
5,C006,[],0
6,C007,['machine learning'],1
7,C008,"['ict network security risks', 'cyber security']",2
8,C009,['sql'],1
9,C010,"['postgresql', 'typescript']",2
